In [ ]:
!pip install transformers datasets torchmetrics

In [ ]:
import torch
import numpy as np

from datasets import load_dataset, Audio, DatasetDict

from transformers import AutoModel, AutoModelForAudioClassification, AutoFeatureExtractor

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Χρήση συσκευής: {device}")

In [ ]:
model_or = AutoModel.from_pretrained("microsoft/wavlm-base").to(device).eval()
model_ft = AutoModelForAudioClassification.from_pretrained("VasilisAsim/wavlm-finetuned-cremad").to(device).eval()
feature_extractor = AutoFeatureExtractor.from_pretrained("VasilisAsim/wavlm-finetuned-cremad")

In [ ]:
from datasets import load_dataset, Audio

#Φόρτωση του dataset, από το HuggingFace
dataset = load_dataset("AbstractTTS/CREMA-D", split="train")

#Μετατροπή συχνότητας σε 16kHz
dataset = dataset.cast_column("audio", Audio(sampling_rate=16_000))

print(dataset.features)
print(dataset[0]["file"], dataset[0]["major_emotion"])

In [ ]:
labels = sorted(set(dataset["major_emotion"]))
num_labels = len(labels)
label2id = {name: i for i, name in enumerate(labels)}
id2label = {i: name for i, name in enumerate(labels)}

def add_speaker(batch):
    batch["actor_id"] = batch["file"].split("_")[0]
    return batch

dataset = dataset.map(add_speaker)
print(len(set(dataset["actor_id"])), "μοναδικοί ομιλητές")

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

actor_ids = dataset["actor_id"]
splitter = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)
train_idx, test_idx = next(splitter.split(X=range(len(dataset)), groups=actor_ids))

from datasets import DatasetDict
dataset = DatasetDict({
    "train": dataset.select(train_idx),
    "test": dataset.select(test_idx)
})

# Έλεγχος ότι δεν υπάρχει επικάλυψη ομιλητών
train_speakers = set(dataset["train"]["actor_id"])
test_speakers = set(dataset["test"]["actor_id"])
print("Επικάλυψη:", train_speakers & test_speakers)  # πρέπει να είναι κενό σύνολο

In [ ]:
def extract_embeddings(model, feature_extractor, dataset, num_samples=150, encoder_attr=None):
    probe_model = getattr(model, encoder_attr) if encoder_attr else model
    num_layers = model.config.num_hidden_layers + 1
    layer_embeddings = [[] for _ in range(num_layers)]

    for i in range(num_samples):
        sample = dataset[i]
        audio = sample["audio"]["array"]
        inputs = feature_extractor(
            audio,
            sampling_rate=16000,
            max_length=6*16000,
            truncation=True,
            return_tensors="pt"
        ).to(device)
        with torch.no_grad():
            outputs = probe_model(inputs["input_values"], output_hidden_states=True)
        for layer, h in enumerate(outputs.hidden_states):
            pooled = h.mean(dim=1)
            layer_embeddings[layer].append(pooled.squeeze().cpu().numpy())

    return [np.array(layer) for layer in layer_embeddings]

In [ ]:
emb_pretrained = extract_embeddings(model_or, feature_extractor, dataset["test"], encoder_attr=None)
emb_finetuned = extract_embeddings(model_ft, feature_extractor, dataset["test"], encoder_attr="wavlm")

del model_or, model_ft
torch.cuda.empty_cache()

In [ ]:
#Συνάρτηση υπολογισμού του CKA
def linear_CKA(X, Y):

    X = X - X.mean(0)
    Y = Y - Y.mean(0)

    hsic = np.linalg.norm(X.T @ Y, ord="fro") ** 2

    var1 = np.linalg.norm(X.T @ X, ord="fro")
    var2 = np.linalg.norm(Y.T @ Y, ord="fro")

    return hsic / (var1 * var2)

In [ ]:
import pandas as pd

num_layers = len(emb_pretrained)
cka_scores = [linear_CKA(emb_pretrained[l], emb_finetuned[l]) for l in range(num_layers)]

cka_results = pd.DataFrame({"layer": range(num_layers), "cka": cka_scores})

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(cka_results["layer"], cka_results["cka"], marker="o", color="#065A82")
plt.xlabel("Layer index (0 = embeddings, 12 = τελευταίο transformer layer)")
plt.ylabel("CKA similarity (Pretrained vs Fine-tuned)")
plt.title("Layer-wise CKA: wavlm πριν/μετά το Fine-tuning (Crema-D)")
plt.ylim(0, 1)
plt.grid(alpha=0.3)
plt.show()